# Data Transformation

**Focus:** Flatten event payloads into analytics-ready tables using ETL pipelines.

**Source File:** case.json

# Notebook Overview

This notebook implements a simplified Medallion Architecture (Bronze → Silver → Gold) using SQLite and JSON source events.

The objective is to demonstrate:

- Raw data ingestion
- Business-driven flattening
- Data quality validation
- Layered transformations
- Aggregated analytical datasets
- CSV export generation

---

# Notebook Structure

## 1. Environment Setup

### Components Notes

The implementation intentionally uses lightweight Python components to keep the solution self-contained and easy to review.

| Component | Technology | Notes |
|------------|------------|---------|
| Database | `sqlite3` | Built-in Python library. Used to simulate Bronze, Silver, and Gold tables without requiring an external database server. |
| JSON Processing | `json` | Built-in Python library used to parse source payloads. |
| Date/Time Handling | `datetime` | Built-in Python library used for UTC → Brazil time (UTC-3) conversion and audit timestamps. |
| CSV Export | `csv` | Built-in Python library used to generate the required business output files. |
| Reporting Preview | `pandas` | Used only for displaying exported CSV files inside the notebook and improving readability during review. |
| ETL Logic | Python Functions | Bronze ingestion, Silver transformations, Gold aggregations, DQ validation, and exports are implemented as reusable functions. |

---

## 2. Table Definitions

All schemas are defined centrally in a metadata-driven dictionary.

**Why this approach instead of hardcoded SQL statements ?**

Benefits
- Centralized schema management
- Reduced code duplication
- Easier onboarding of new tables
- Consistent table creation process

For a single schema change, the effort may look similar. The real benefit appears when managing dozens of tables, enforcing standards, generating documentation, or migrating to platforms like Microsoft Fabric. Metadata allows schema information to be reused across multiple processes instead of being embedded in SQL code.

For example, if we want to check whether the ingested_at column exists in all tables, we can simply use a for loop instead of inspecting every SQL table definition.
```
for table_name, schema in schemas.items():
column_names = [col[0] for col in schema["columns"]]
 
assert "ingested_at" in column_names, (
f"{table_name} is missing required column: ingested_at"
)
```

### Layers:

#### Bronze

- bronze_raw_events

#### Silver

- silver_dynamic_price_option
- silver_dynamic_price_range
- silver_curated_offer_options

#### Data Quality

- silver_rejected_events
- dq_results

### Gold

- gold_offer_pricing_summary
- gold_dealer_performance

---

# Execution Flow

Instead of encapsulating the entire workflow into a single `main()` function, this notebook focus on separating each ETL stage into individual cells.
This approach makes the processing flow easier to understand, and demonstrate during design discussions.
The notebook should be executed in the following order:

```python
load_bronze_raw(...)
# Read source JSON files
# Generate event_id and payload_hash
# Store raw payloads unchanged
# Load bronze_raw_events
 
bronze_report(...)
# Validate Bronze ingestion
# Display record counts
# Show duplicate statistics
# Preview raw events
 
 
process_silver(...)
# Parse JSON payloads
# Apply DQ validation rules
# Convert UTC → Brazil Time (UTC-3)
# Flatten nested structures
# Build Silver business entities
# Store rejected records
 
silver_report(...)
# Audit Silver processing
# Display processed vs rejected records
# Show DQ failure counts
# Preview Silver tables
 
 
create_gold_tables(...)
# Create analytics-ready Gold table schemas
# Define reporting structures
# Prepare Gold storage layer
 
load_gold(...)
# Aggregate Silver entities
# Generate business KPIs
# Build offer-level and dealer-level summaries
# Populate Gold tables
 
gold_report(...)
# Validate Gold outputs
# Display KPI row counts
# Preview aggregated business metrics
 
 
export_csv(...)
# Generate required business deliverables
# Export DynamicPriceOption.csv
# Export DynamicPriceRange.csv
# Export CuratedOfferOptions.csv
# Apply required formatting rules
 
preview_exports(...)
# Read generated CSV files
# Display row counts
# Display column counts
# Preview first 5 records of each file
```

---


# CSV Export

Function:

```python
export_csv() #Generated files 
preview_exports() # Review export process and sample data
```


# Data Quality Controls

Implemented DQ rules:

- DQ001_MALFORMED_JSON
- DQ002_MISSING_REQUIRED_FIELDS
- DQ004_INVALID_PRICE_RANGE
- DQ005_NEGATIVE_PRICE
- DQ007_SCHEMA_CHANGE

Rejected records are stored in:

```text
silver_rejected_events
```

for auditability and troubleshooting.

---

# Notes for Reviewers

This notebook is intended as a proof-of-concept implementation.

The focus is on:

- Data modeling
- Layered transformations
- Data quality validation
- Business flattening logic
- Reporting outputs

Operational concerns such as scheduling, monitoring, orchestration, alerting, metadata governance, and distributed processing are discussed in the final assessment section.

In [1]:
from pathlib import Path
import json
import sqlite3
import os
import hashlib
import json
from datetime import datetime, timezone, timedelta
import pandas as pd

BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
DATA_FILE =  Path("case.json")
FILE_NAME = 'case.json'
DB_PATH = DATA_DIR / 'transformetl.db'

conn = sqlite3.connect(DB_PATH)
print(conn)
cur = conn.cursor()

def generate_event_id(event):
    key = (
        event["EventName"]
        + event["EnqueuedTimeUtc"]
        + event["Payload"]
    )
    return hashlib.sha256(key.encode()).hexdigest()

def get_partition_date(event):
    """
    Partition by event date.
    Example: 2021-09-05
    """
    return event["EnqueuedTimeUtc"].split(" ")[0]

#UTC + 3 
def utc_to_brt(utc_text):
    utc_dt = datetime.strptime(
        utc_text,
        "%Y-%m-%d %H:%M:%S UTC"
    )

    br_dt = utc_dt - timedelta(hours=3)

    return br_dt.strftime("%Y-%m-%d %H:%M:%S")
    

#### Quick observation this payload json to see how we flatten this into 3 silver tables 
Business context:
 
- Dynamic Price answers: "What price should we recommend?"
  ApplyDynamicPriceRange → recommended price range for an offer.
  ApplyDynamicPricePerOption → recommended price for each option.
 
- Curate Offer answers: "Which option should we choose?"
  Compares dealer options using price, ETA, availability, etc.
  Produces winner/loser options and defeat reasons.

Look at this simplified json:   
```
{
"provider": "ApplyDynamicPricePerOption",
"offerId": "00991873...",
"algorithmOutput": [
    {"uniqueOptionId":"A","bestPrice":92.45},
    {"uniqueOptionId":"B","bestPrice":92.45},
    {"uniqueOptionId":"C","bestPrice":92.45}
  ]
},
{
"provider":"ApplyDynamicPriceRange",
"offerId":"329194f3...",
"algorithmOutput":{
    "min_global":85,
    "min_recommended":87.2,
    "max_recommended":97.65
  }
},
{
"curationProvider": "ByPrice",
"offerId": "0a06a478-77b3-4941-83d3-84ded18d4731",
"dealerId": "6517",
"options": [
  {
    "uniqueOptionId": "b0e296a9-0590-f0e0-8211-243a2ededb12",
    "optionId": "6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2",
    "isMobileDealer": true
    ...
  }  
}
```
- **ApplyDynamicPricePerOption** → the payload describes multiple options inside one offer (e.g. `offerId = 00991873-194e-4a6e-89c9-8f68668b6aaa` contains 8 `uniqueOptionId`s). Each option has its own `bestPrice`, so each option becomes a separate row in `silver_dynamic_price_option`.
- **ApplyDynamicPriceRange** → the payload describes a single pricing range for one offer (e.g. `offerId = 329194f3-95a4-45ef-b3d0-2796f74ce2a0`) → **1 offer = 1 row** in `silver_dynamic_price_range`.
- **CurateOffer** the payload describes multiple dealers inside one offer, and each dealer contains multiple options. Since each dealer-option combination has its own price, ETA, winner flag, defeat reasons, etc., each dealer option becomes a separate row in `silver_curated_offer_options`


In [2]:
# Define metadata array, not commit to sqllite3 yet
schemas = {
    # ==========================================================
    # Bronze stores the raw payload unchanged 
    # with audit metadata (`source_file`, `batch_id`, `ingested_at_utc`, `ingestion_date`) 
    # to support detection.
    # ==========================================================
    "bronze_raw_events": {
        "columns": [
            ("event_id", "TEXT PRIMARY KEY"),
            ("event_name", "TEXT NOT NULL"),
            ("enqueued_time_utc", "TEXT NOT NULL"),
            ("payload_raw", "TEXT NOT NULL"),
            ("source_file", "TEXT"),
            ("batch_id", "TEXT"),
            ("ingested_at_utc", "TEXT"),
            ("ingestion_date", "TEXT")
        ]
    },

    # ==========================================================
    # SILVER: as declared above
    # ==========================================================
    "silver_curated_offer_options": {
        "columns": [
            ("event_id", "TEXT PRIMARY KEY"),
            ("curation_provider", "TEXT"),
            ("offer_id", "TEXT NOT NULL"),
            ("dealer_id", "TEXT"),
            ("unique_option_id", "TEXT"),
            ("option_id", "TEXT"),
            ("is_mobile_dealer", "INTEGER"),
            ("is_open", "INTEGER"),
            ("eta", "TEXT"),
            ("chama_score", "REAL"),
            ("product_brand", "TEXT"),
            ("is_winner", "INTEGER"),
            ("minimum_price", "REAL"),
            ("maximum_price", "REAL"),
            ("dynamic_price", "REAL"),
            ("final_price", "REAL"),
            ("defeat_primary_reason", "TEXT"),
            ("defeat_reasons", "TEXT"),
            ("event_time_brt", "TEXT"),
            ("processed_at_utc", "TEXT")
        ]
    },

    "silver_dynamic_price_option": {
        "columns": [
            ("event_id", "TEXT PRIMARY KEY"),
            ("provider", "TEXT"),
            ("offer_id", "TEXT NOT NULL"),
            ("unique_option_id", "TEXT"),
            ("best_price", "REAL"),
            ("event_time_brt", "TEXT"),
            ("processed_at_utc", "TEXT")
        ]
    },

    "silver_dynamic_price_range": {
        "columns": [
            ("event_id", "TEXT PRIMARY KEY"),
            ("provider", "TEXT"),
            ("offer_id", "TEXT NOT NULL"),
            ("min_global", "REAL"),
            ("min_recommended", "REAL"),
            ("max_recommended", "REAL"),
            ("difference_min_recommend_min_theory", "REAL"),
            ("event_time_brt", "TEXT"),
            ("processed_at_utc", "TEXT")
        ]
    },

    # ==========================================================
    # DATA QUALITY / ERROR HANDLING 
    # Validate incoming data, reject invalid records, and store failures, bad data to review
    # ==========================================================
    "silver_rejected_events": {
        "columns": [
            ("event_id", "TEXT"),
            ("event_name", "TEXT"),
            ("payload_raw", "TEXT"),
            ("error_reason", "TEXT"),
            ("rejected_at_utc", "TEXT"),
            ("error_code", "TEXT")
        ]
    },

    "dq_results": {
        "columns": [
            ("run_id", "TEXT"),
            ("table_name", "TEXT"),
            ("check_name", "TEXT"),
            ("status", "TEXT"),
            ("failed_count", "INTEGER"),
            ("checked_at_utc", "TEXT")
        ]
    }
}

In [3]:
# For simplicity, this notebook recreates tables during execution
# In a production environment, these tables would be retained...
# ..and continuously updated rather than recreated.
for table_name in schemas.keys():

    conn.execute(
        f"DROP TABLE IF EXISTS {table_name}"
    )

    print(f"Dropped: {table_name}")

conn.commit()

Dropped: bronze_raw_events
Dropped: silver_curated_offer_options
Dropped: silver_dynamic_price_option
Dropped: silver_dynamic_price_range
Dropped: silver_rejected_events
Dropped: dq_results


In [4]:
# Create physical tables in database
for table_name, schema in schemas.items():

    columns_sql = ",\n    ".join(
        f"{name} {dtype}"
        for name, dtype in schema["columns"]
    )

    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        {columns_sql}
    );
    """
    conn.execute(create_sql)

conn.commit()

# BRONZE PROCESS
Partition Strategy
------------------
Bronze data is partitioned by ingestion_date (YYYY-MM-DD).
prepare for: 
```
CREATE TABLE bronze_raw_events
USING DELTA
PARTITIONED BY (ingestion_date)
AS ...
```
Benefits:
- Faster incremental processing
- Efficient date-based filtering
- Easier replay and retention management

In [5]:
def load_bronze_raw(conn, json_file_path, batch_id=None):
    """
    Bronze layer:
    - Preserve original payload
    - Partition by ingestion_date
    - Support idempotent reloads
    """
    source_file = json_file_path.name if hasattr(json_file_path, "name") else str(json_file_path)

    with open(json_file_path, "r", encoding="utf-8") as f:
        events = json.load(f)

    now_utc = datetime.now(timezone.utc).isoformat()

    rows = [
        (
            generate_event_id(event),
            event["EventName"],
            event["EnqueuedTimeUtc"],
            event["Payload"],  # keep raw payload
            source_file,
            batch_id,
            now_utc,
            get_partition_date(event), #prepare for parition in future implement
        )
        for event in events
    ]

    conn.executemany("""
        INSERT OR IGNORE INTO bronze_raw_events (
            event_id, event_name, enqueued_time_utc, payload_raw,
            source_file, batch_id, ingested_at_utc, ingestion_date
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, rows)

    conn.commit()
    print(f"Loaded {len(rows)} events")

In [6]:
def bronze_report(conn):
    """Generate Bronze layer summary report."""
    print("=" * 80)
    print("BRONZE LAYER REPORT")
    print("=" * 80)

    # --------------------------------------------------
    # Record count
    # --------------------------------------------------
    total_records = conn.execute(
        "SELECT COUNT(*) FROM bronze_raw_events"
    ).fetchone()[0]
    print(f"\nTotal Records: {total_records}")

    # --------------------------------------------------
    # Partition summary
    # --------------------------------------------------
    print("\nPartition Distribution")
    partitions = pd.read_sql_query("""
        SELECT ingestion_date, COUNT(*) AS record_count
        FROM bronze_raw_events
        GROUP BY ingestion_date
        ORDER BY ingestion_date
    """, conn)
    print(partitions.to_string(index=False))

    # --------------------------------------------------
    # Source files
    # --------------------------------------------------
    print("\nSource Files")
    files = pd.read_sql_query("""
        SELECT source_file, COUNT(*) AS records
        FROM bronze_raw_events
        GROUP BY source_file
    """, conn)
    print(files.to_string(index=False))

    # --------------------------------------------------
    # Sample rows
    # --------------------------------------------------
    print("\nSample Data (Top 5 Rows)")
    sample = pd.read_sql_query("""
        SELECT
            event_name,
            enqueued_time_utc,
            substr(payload_raw, 1, 80) || '...' AS payload_preview
        FROM bronze_raw_events
        LIMIT 5
    """, conn)
    print(sample.to_string(index=False))

    print("\n" + "=" * 80)

In [7]:
# Run load and preview process quality
load_bronze_raw(conn, FILE_NAME)
bronze_report(conn)

Loaded 37 events
BRONZE LAYER REPORT

Total Records: 36

Partition Distribution
ingestion_date  record_count
    2021-08-18             6
    2021-08-25             5
    2021-09-05            25

Source Files
source_file  records
  case.json       36

Sample Data (Top 5 Rows)
         event_name       enqueued_time_utc                                                                     payload_preview
DynamicPrice_Result 2021-09-05 08:04:08 UTC {"provider":"ApplyDynamicPriceRange","offerId":"a6611d55-9624-4381-8cdd-323ee368...
DynamicPrice_Result 2021-08-18 11:43:23 UTC {"provider":"ApplyDynamicPricePerOption","offerId":"56e0702c-0218-4626-8d3d-ae9d...
DynamicPrice_Result 2021-09-05 09:04:04 UTC {"provider":"ApplyDynamicPriceRange","offerId":"b8c636fa-8241-47dc-ac40-bdf438a0...
 CurateOffer_Result 2021-08-25 05:02:55 UTC [{"curationProvider":"ByPrice","offerId":"149f0e53-ff85-425f-a01a-8710f06704ea",...
DynamicPrice_Result 2021-09-05 08:03:28 UTC {"provider":"ApplyDynamicPriceRange","

# SILVER PROCESS
This section is a bit long, hope you don't mind scrolling manually.

In [8]:
def reject_event(conn, event_id, event_name, payload_raw, error_code, error_reason):
    conn.execute("""
        INSERT INTO silver_rejected_events (
            event_id, event_name, error_code, error_reason, payload_raw, rejected_at_utc
        ) VALUES (?, ?, ?, ?, ?, ?)
    """, (
        event_id,
        event_name,
        error_code,
        error_reason,
        payload_raw,
        datetime.now(timezone.utc).isoformat()
    ))

In [9]:
def process_silver(conn):
    bronze_events = conn.execute("""
        SELECT event_id, event_name, enqueued_time_utc, payload_raw
        FROM bronze_raw_events
    """).fetchall()

    processed = 0
    rejected = 0
    skipped = 0

    for event_id, event_name, enqueued_time_utc, payload_raw in bronze_events:
        try:
            payload = json.loads(payload_raw)
            event_time_brt = utc_to_brt(enqueued_time_utc)
            processed_at_utc = datetime.now(timezone.utc).isoformat()

            # ==================================================
            # CURATE OFFER RESULT
            # ==================================================
            if isinstance(payload, list):
                for offer in payload:
                    curation_provider = offer.get("curationProvider")
                    offer_id = offer.get("offerId")
                    dealer_id = offer.get("dealerId")
                    options = offer.get("options", [])

                    for option in options:
                        unique_option_id = option.get("uniqueOptionId")
                        silver_event_id = f"{event_id}_{unique_option_id}"

                        exists = conn.execute(
                            "SELECT 1 FROM silver_curated_offer_options WHERE event_id = ?",
                            (silver_event_id,)
                        ).fetchone()

                        if exists:
                            skipped += 1
                            continue

                        conn.execute("""
                            INSERT INTO silver_curated_offer_options (
                                event_id, curation_provider, offer_id, dealer_id,
                                unique_option_id, option_id, is_mobile_dealer, is_open,
                                eta, chama_score, product_brand, is_winner,
                                minimum_price, maximum_price, dynamic_price, final_price,
                                defeat_primary_reason, defeat_reasons, event_time_brt, processed_at_utc
                            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """, (
                            silver_event_id,
                            curation_provider,
                            offer_id,
                            dealer_id,
                            unique_option_id,
                            option.get("optionId"),
                            option.get("isMobileDealer"),
                            option.get("isOpen"),
                            option.get("eta"),
                            option.get("chamaScore"),
                            option.get("productBrand"),
                            option.get("isWinner"),
                            option.get("minimumPrice"),
                            option.get("maximumPrice"),
                            option.get("dynamicPrice"),
                            option.get("finalPrice"),
                            option.get("defeatPrimaryReason", ""),
                            str(option.get("defeatReasons", [])),
                            event_time_brt,
                            processed_at_utc
                        ))

                processed += 1
                continue

            # ==================================================
            # DYNAMIC PRICE PAYLOAD
            # ==================================================
            required_fields = ["provider", "offerId", "algorithmOutput"]
            missing_fields = [f for f in required_fields if f not in payload]

            if missing_fields:
                reject_event(
                    conn, event_id, event_name, payload_raw,
                    "DQ002_MISSING_REQUIRED_FIELDS",
                    f"Missing fields: {missing_fields}"
                )
                rejected += 1
                continue

            provider = payload["provider"]
            offer_id = payload["offerId"]

            # ==================================================
            # ApplyDynamicPricePerOption
            # ==================================================
            if provider == "ApplyDynamicPricePerOption":
                outputs = payload["algorithmOutput"]

                if not isinstance(outputs, list):
                    reject_event(
                        conn, event_id, event_name, payload_raw,
                        "DQ007_SCHEMA_CHANGE",
                        "algorithmOutput must be an array"
                    )
                    rejected += 1
                    continue

                for item in outputs:
                    unique_option_id = item.get("uniqueOptionId")
                    best_price = item.get("bestPrice")

                    if unique_option_id is None or best_price is None:
                        continue

                    best_price = float(best_price)

                    if best_price < 0:
                        reject_event(
                            conn, event_id, event_name, payload_raw,
                            "DQ005_NEGATIVE_PRICE",
                            f"Negative best_price: {best_price}"
                        )
                        rejected += 1
                        continue

                    silver_event_id = f"{event_id}_{unique_option_id}"
                    exists = conn.execute(
                        "SELECT 1 FROM silver_dynamic_price_option WHERE event_id = ?",
                        (silver_event_id,)
                    ).fetchone()

                    if exists:
                        skipped += 1
                        continue

                    conn.execute("""
                        INSERT INTO silver_dynamic_price_option (
                            event_id, provider, offer_id, unique_option_id,
                            best_price, event_time_brt, processed_at_utc
                        ) VALUES (?, ?, ?, ?, ?, ?, ?)
                    """, (
                        silver_event_id, provider, offer_id, unique_option_id,
                        best_price, event_time_brt, processed_at_utc
                    ))

                processed += 1

            # ==================================================
            # ApplyDynamicPriceRange
            # ==================================================
            elif provider == "ApplyDynamicPriceRange":
                algo = payload["algorithmOutput"]

                min_global = float(algo["min_global"])
                min_recommended = float(algo["min_recommended"])
                max_recommended = float(algo["max_recommended"])
                diff = float(algo["differenceMinRecommendMinTheory"])

                if (
                    min_global < 0
                    or min_recommended < min_global
                    or max_recommended < min_recommended
                ):
                    reject_event(
                        conn, event_id, event_name, payload_raw,
                        "DQ004_INVALID_PRICE_RANGE",
                        f"min_global={min_global}, min_recommended={min_recommended}, max_recommended={max_recommended}"
                    )
                    rejected += 1
                    continue

                exists = conn.execute(
                    "SELECT 1 FROM silver_dynamic_price_range WHERE event_id = ?",
                    (event_id,)
                ).fetchone()

                if exists:
                    skipped += 1
                    continue

                conn.execute("""
                    INSERT INTO silver_dynamic_price_range (
                        event_id, provider, offer_id, min_global,
                        min_recommended, max_recommended,
                        difference_min_recommend_min_theory,
                        event_time_brt, processed_at_utc
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    event_id, provider, offer_id, min_global,
                    min_recommended, max_recommended, diff,
                    event_time_brt, processed_at_utc
                ))

                processed += 1

            else:
                reject_event(
                    conn, event_id, event_name, payload_raw,
                    "DQ007_SCHEMA_CHANGE",
                    f"Unsupported provider: {provider}"
                )
                rejected += 1

        except json.JSONDecodeError:
            reject_event(
                conn, event_id, event_name, payload_raw,
                "DQ001_MALFORMED_JSON",
                "Payload is not valid JSON"
            )
            rejected += 1

        except Exception as ex:
            reject_event(
                conn, event_id, event_name, payload_raw,
                "DQ999_UNEXPECTED_ERROR",
                str(ex)
            )
            rejected += 1

    conn.commit()

    print(
        f"[SILVER] processed={processed}, "
        f"rejected={rejected}, "
        f"skipped_duplicates={skipped}"
    )

In [10]:
def silver_report(conn):
    print("\n" + "=" * 80)
    print("SILVER INGESTION REPORT")
    print("=" * 80)

    # ==================================================
    # Silver Output Summary
    # ==================================================
    print("\nSilver Tables")
    tables = [
        "silver_dynamic_price_option",
        "silver_dynamic_price_range",
        "silver_curated_offer_options",
    ]
    for table in tables:
        try:
            cnt = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
            print(f"  {table:<35}: {cnt}")
        except Exception:
            print(f"  {table:<35}: NOT FOUND")

    # ==================================================
    # DQ Summary
    # ==================================================
    print("\nData Quality Summary")
    dq_summary = {
        "DQ001_MALFORMED_JSON": 0,
        "DQ002_MISSING_REQUIRED_FIELDS": 0,
        "DQ003_DUPLICATE_EVENT": 0,
        "DQ004_INVALID_PRICE_RANGE": 0,
        "DQ005_NEGATIVE_PRICE": 0,
        "DQ006_LATE_ARRIVING_DATA": 0,
        "DQ007_SCHEMA_CHANGE": 0,
    }

    rows = conn.execute("""
        SELECT COALESCE(error_code, 'UNKNOWN'), COUNT(*)
        FROM silver_rejected_events
        GROUP BY 1
    """).fetchall()

    for code, count in rows:
        dq_summary[code] = count

    for code, count in dq_summary.items():
        icon = "✅" if count == 0 else "❌"
        print(f"  {icon} {code:<35} {count}")

    # ==================================================
    # Failed Records
    # ==================================================
    rejected_count = conn.execute(
        "SELECT COUNT(*) FROM silver_rejected_events"
    ).fetchone()[0]

    print("\nFailed Records")
    if rejected_count == 0:
        print("  ✅ No rejected records")
    else:
        failed_rows = conn.execute("""
            SELECT event_id, COALESCE(error_code, 'UNKNOWN'), error_reason
            FROM silver_rejected_events
            ORDER BY rejected_at_utc
        """).fetchall()

        for idx, (event_id, error_code, error_reason) in enumerate(failed_rows, start=1):
            print(f"\n[{idx}] {error_code}")
            print(f"    Event ID : {event_id}")
            print(f"    Reason   : {error_reason}")

    # ==================================================
    # Processing Summary
    # ==================================================
    bronze_count = conn.execute(
        "SELECT COUNT(*) FROM bronze_raw_events"
    ).fetchone()[0]

    print("\nProcessing Summary")
    print(f"  Bronze Records     : {bronze_count}")
    print(f"  Rejected Records   : {rejected_count}")
    print(f"  Successful Records : {bronze_count - rejected_count}")
    print("\n" + "=" * 80)

In [11]:
# Run load and review report of process quality
process_silver(conn)
 
silver_report(conn)

[SILVER] processed=36, rejected=0, skipped_duplicates=0

SILVER INGESTION REPORT

Silver Tables
  silver_dynamic_price_option        : 48
  silver_dynamic_price_range         : 25
  silver_curated_offer_options       : 40

Data Quality Summary
  ✅ DQ001_MALFORMED_JSON                0
  ✅ DQ002_MISSING_REQUIRED_FIELDS       0
  ✅ DQ003_DUPLICATE_EVENT               0
  ✅ DQ004_INVALID_PRICE_RANGE           0
  ✅ DQ005_NEGATIVE_PRICE                0
  ✅ DQ006_LATE_ARRIVING_DATA            0
  ✅ DQ007_SCHEMA_CHANGE                 0

Failed Records
  ✅ No rejected records

Processing Summary
  Bronze Records     : 36
  Rejected Records   : 0
  Successful Records : 36



## Assessment of Current Silver Processing Design
 
The current `process_silver()` implementation successfully demonstrates the main Silver-layer responsibilities:
 
- Payload parsing
- Schema validation
- Data quality checks
- Duplicate protection
- Business-level flattening
- Rejected record handling
 
For the current dataset size, the solution is simple, readable, and effective.

## Current Limitations
| Area | Limitation |
|--------|-----------|
| Freshness Validation | Not been established yet  |
| Schema Evolution | New payload versions may require code changes. |
| Metadata Governance | Validation rules are embedded in notebook code rather than managed centrally. |
| Single-Threaded Processing | Events are processed sequentially using Python loops. |
| Scaling | Not designed for millions of events per day. |
| DQ Metrics | Rejected records are stored, but DQ statistics are not persisted in a dedicated audit table. |
 
---

#### Potential impact:
Long notebook execution times
Increased memory pressure
Slow duplicate checks
Gold aggregation bottlenecks
CSV generation delays
Pipeline schedule overlap


| Component | Current State | Future State |
| :--- | :--- | :--- |
| **Storage** | SQLite | Delta Lake / Fabric Lakehouse |
| **Processing Engine** | Python Notebook | Spark Notebook |
| **Duplicate Detection** | Row-by-row lookup | Delta MERGE / Constraints |
| **Validation Rules** | Hardcoded | Metadata-driven DQ framework |
| **Freshness Checks** | x | Automated force retry |
| **Schema Changes** | Manual code updates | Schema registry and versioning |
| **Gold Aggregation** | Full rebuild | Incremental MERGE processing |
| **Monitoring** | Notebook reports | Fabric Monitoring Hub / Azure Monitor |
| **DQ Reporting** | Rejected records only | Dedicated `dq_audit_results` table |

# Gold Layer

## Purpose

The Gold layer contains business-ready datasets optimized for reporting, dashboarding, and analytics.

All Gold tables are built from validated Silver data.

---

# Data Lineage

```text
bronze_raw_events
        │
        ▼
silver_dynamic_price_option
        │
        ├──────────────► gold_offer_daily_summary
        │
        └──────────────► gold_brand_daily_summary

```

---

# Gold Table 1

## gold_offer_daily_summary

### Grain

```text
1 row = 1 offer_id per day
```

### Purpose

Provides daily offer-level pricing statistics.

### Source

```text
silver_dynamic_price_option
```

### Metrics

- option_count
- avg_best_price
- min_best_price
- max_best_price

### Query

```sql
SELECT
    offer_id,
    event_date_brt,
    COUNT(*) AS option_count,
    AVG(best_price) AS avg_best_price,
    MIN(best_price) AS min_best_price,
    MAX(best_price) AS max_best_price
FROM silver_dynamic_price_option
GROUP BY
    offer_id,
    event_date_brt;
```

---

# Gold Table 2

## gold_brand_daily_summary

### Grain

```text
1 row = 1 product_brand per day
```

### Purpose

Provides daily pricing and competition statistics by brand.

### Source

```text
silver_dynamic_price_option
```

### Metrics

- option_count
- avg_price
- winning_option_count

### Query

```sql
SELECT
    event_date_brt,
    product_brand,
    COUNT(*) AS option_count,
    AVG(best_price) AS avg_price,
    SUM(is_winner) AS winning_option_count
FROM silver_dynamic_price_option
GROUP BY
    event_date_brt,
    product_brand;
```
 

In [12]:
def create_gold_tables(conn):

    gold_tables = {

        "gold_offer_pricing_summary": {
            "description":
                "Daily offer-level pricing KPIs aggregated from option pricing.",
            "grain":
                "1 row per offer per day",
            "source":
                "silver_dynamic_price_option",
            "columns": [
                ("offer_id", "TEXT"),
                ("event_date_brt", "TEXT"),
                ("option_count", "INTEGER"),
                ("avg_best_price", "REAL"),
                ("min_best_price", "REAL"),
                ("max_best_price", "REAL")
            ]
        },

        "gold_dealer_performance": {
            "description":
                "Daily dealer performance KPIs aggregated from curated offer options.",
            "grain":
                "1 row per dealer per day",
            "source":
                "silver_curated_offer_options",
            "columns": [
                ("dealer_id", "TEXT"),
                ("product_brand", "TEXT"),
                ("event_date_brt", "TEXT"),
                ("offer_count", "INTEGER"),
                ("winner_count", "INTEGER"),
                ("winner_rate", "REAL"),
                ("avg_final_price", "REAL"),
                ("avg_chama_score", "REAL")
            ]
        }
    }

    for table_name, metadata in gold_tables.items():

        conn.execute(
            f"DROP TABLE IF EXISTS {table_name}"
        )

        columns_sql = ",\n".join(
            f"{col_name} {datatype}"
            for col_name, datatype
            in metadata["columns"]
        )

        conn.execute(
            f"""
            CREATE TABLE {table_name}
            (
                {columns_sql}
            )
            """
        )

        print(
            f"[GOLD] Created: {table_name}"
        )
        print(
            f"        Grain : {metadata['grain']}"
        )
        print(
            f"        Source: {metadata['source']}"
        )

    conn.commit()

    print(
        "[GOLD] metadata and tables created successfully"
    )

In [13]:
def load_gold(conn):
    print("\n[GOLD] Loading aggregates...")

    # ==================================================
    # Clean Gold Tables
    # ==================================================
    conn.execute("DELETE FROM gold_offer_pricing_summary")
    conn.execute("DELETE FROM gold_dealer_performance")

    # ==================================================
    # GOLD 1: gold_offer_pricing_summary
    # Grain: 1 row per offer per day
    # Source: silver_dynamic_price_option
    # ==================================================
    conn.execute("""
        INSERT INTO gold_offer_pricing_summary (
            offer_id,
            event_date_brt,
            option_count,
            avg_best_price,
            min_best_price,
            max_best_price
        )
        SELECT
            offer_id,
            substr(event_time_brt, 1, 10) AS event_date_brt,
            COUNT(*) AS option_count,
            ROUND(AVG(best_price), 2) AS avg_best_price,
            MIN(best_price) AS min_best_price,
            MAX(best_price) AS max_best_price
        FROM silver_dynamic_price_option
        GROUP BY
            offer_id,
            substr(event_time_brt, 1, 10)
    """)

    # ==================================================
    # GOLD 2: gold_dealer_performance
    # Grain: 1 row per dealer / brand / day
    # Source: silver_curated_offer_options
    # ==================================================
    conn.execute("""
        INSERT INTO gold_dealer_performance (
            dealer_id,
            product_brand,
            event_date_brt,
            offer_count,
            winner_count,
            winner_rate,
            avg_final_price,
            avg_chama_score
        )
        SELECT
            dealer_id,
            product_brand,
            substr(event_time_brt, 1, 10) AS event_date_brt,
            COUNT(*) AS offer_count,
            SUM(CASE WHEN is_winner = 1 THEN 1 ELSE 0 END) AS winner_count,
            ROUND(100.0 * SUM(CASE WHEN is_winner = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS winner_rate,
            ROUND(AVG(final_price), 2) AS avg_final_price,
            ROUND(AVG(chama_score), 2) AS avg_chama_score
        FROM silver_curated_offer_options
        GROUP BY
            dealer_id,
            product_brand,
            substr(event_time_brt, 1, 10)
    """)

    conn.commit()

    pricing_rows = conn.execute(
        "SELECT COUNT(*) FROM gold_offer_pricing_summary"
    ).fetchone()[0]

    dealer_rows = conn.execute(
        "SELECT COUNT(*) FROM gold_dealer_performance"
    ).fetchone()[0]

    print(f"[GOLD] gold_offer_pricing_summary : {pricing_rows}")
    print(f"[GOLD] gold_dealer_performance    : {dealer_rows}")
    print("[GOLD] Load completed")

In [14]:
def gold_report(conn):

    print("\n" + "=" * 80)
    print("GOLD LAYER AUDIT REPORT")
    print("=" * 80)

    gold_tables = [
        "gold_offer_pricing_summary",
        "gold_dealer_performance"
    ]

    total_rows = 0

    print("\nGold Tables")

    for table in gold_tables:

        count = conn.execute(
            f"SELECT COUNT(*) FROM {table}"
        ).fetchone()[0]

        total_rows += count

        print(
            f"  {table:<35}: {count}"
        )

    print("\nLineage")

    print(
        "  silver_dynamic_price_option"
    )
    print(
        "      ↓"
    )
    print(
        "  gold_offer_pricing_summary"
    )

    print()

    print(
        "  silver_curated_offer_options"
    )
    print(
        "      ↓"
    )
    print(
        "  gold_dealer_performance"
    )

    print(
        f"\nTotal Gold Rows : {total_rows}"
    )

    print("\n" + "-" * 80)
    print("SAMPLE DATA (TOP 5 ROWS)")
    print("-" * 80)

    for table_name in gold_tables:

        print(f"\n[{table_name}]")

        columns = conn.execute(
            f"""
            PRAGMA table_info({table_name})
            """
        ).fetchall()

        headers = [
            col[1]
            for col in columns
        ]

        rows = conn.execute(
            f"""
            SELECT *
            FROM {table_name}
            LIMIT 5
            """
        ).fetchall()

        if not rows:
            print("No data")
            continue

        print(" | ".join(headers))
        print("-" * 120)

        for row in rows:
            print(
                " | ".join(
                    str(value)
                    for value in row
                )
            )

    print("\n" + "=" * 80)

In [15]:
# Create, run load and check process quality
create_gold_tables(conn)
load_gold(conn)
gold_report(conn)

[GOLD] Created: gold_offer_pricing_summary
        Grain : 1 row per offer per day
        Source: silver_dynamic_price_option
[GOLD] Created: gold_dealer_performance
        Grain : 1 row per dealer per day
        Source: silver_curated_offer_options
[GOLD] metadata and tables created successfully

[GOLD] Loading aggregates...
[GOLD] gold_offer_pricing_summary : 6
[GOLD] gold_dealer_performance    : 7
[GOLD] Load completed

GOLD LAYER AUDIT REPORT

Gold Tables
  gold_offer_pricing_summary         : 6
  gold_dealer_performance            : 7

Lineage
  silver_dynamic_price_option
      ↓
  gold_offer_pricing_summary

  silver_curated_offer_options
      ↓
  gold_dealer_performance

Total Gold Rows : 13

--------------------------------------------------------------------------------
SAMPLE DATA (TOP 5 ROWS)
--------------------------------------------------------------------------------

[gold_offer_pricing_summary]
offer_id | event_date_brt | option_count | avg_best_price | min_best_

=== GOLD PROCESS DONE ==== 
# CSV EXPORT 
Source will be extract from Silver tables

In [20]:

def export_csv(conn):
    print("\n[EXPORT] Generating CSV files...")

    # ==================================================
    # DynamicPriceOption.csv
    # ==================================================
    query_opt = """
        SELECT
            provider AS Provider,
            offer_id AS OfferId,
            unique_option_id AS UniqueOptionId,
            best_price AS BestPrice,
            strftime('%d/%m/%Y', event_time_brt) AS EnqueuedTimeSP
        FROM silver_dynamic_price_option
    """
    df_opt = pd.read_sql_query(query_opt, conn)
    df_opt.to_csv("DynamicPriceOption.csv", index=False, encoding="utf-8")
    print("[EXPORT] DynamicPriceOption.csv")

    # ==================================================
    # DynamicPriceRange.csv
    # ==================================================
    query_range = """
        SELECT
            provider AS Provider,
            offer_id AS OfferId,
            min_global AS MinGlobal,
            min_recommended AS MinRecommended,
            max_recommended AS MaxRecommended,
            difference_min_recommend_min_theory AS DifferenceMinRecommendMinTheory,
            strftime('%d/%m/%Y', event_time_brt) AS EnqueuedTimeSP
        FROM silver_dynamic_price_range
    """
    df_range = pd.read_sql_query(query_range, conn)
    df_range.to_csv("DynamicPriceRange.csv", index=False, encoding="utf-8")
    print("[EXPORT] DynamicPriceRange.csv")

    # ==================================================
    # CuratedOfferOptions.csv
    # ==================================================
    query_curated = """
        SELECT
            curation_provider AS CurationProvider,
            offer_id AS OfferId,
            dealer_id AS DealerId,
            unique_option_id AS UniqueOptionId,
            option_id AS OptionId,
            is_mobile_dealer AS IsMobileDealer,
            is_open AS IsOpen,
            eta AS Eta,
            chama_score AS ChamaScore,
            product_brand AS ProductBrand,
            is_winner AS IsWinner,
            minimum_price AS MinimumPrice,
            maximum_price AS MaximumPrice,
            dynamic_price AS DynamicPrice,
            final_price AS FinalPrice,
            COALESCE(defeat_primary_reason, '') AS DefeatPrimaryReason,
            COALESCE(defeat_reasons, '') AS DefeatReasons,
            strftime('%d/%m/%Y', event_time_brt) AS EnqueuedTimeSP
        FROM silver_curated_offer_options
    """
    df_curated = pd.read_sql_query(query_curated, conn)
    df_curated.to_csv("CuratedOfferOptions.csv", index=False, encoding="utf-8")
    print("[EXPORT] CuratedOfferOptions.csv")

    print("[EXPORT] Completed")


def preview_exports():
    files = [
        "DynamicPriceOption.csv",
        "DynamicPriceRange.csv",
        "CuratedOfferOptions.csv",
    ]

    print("\n" + "=" * 100)
    print("CSV EXPORT PREVIEW")
    print("=" * 100)

    for file in files:
        print(f"\n[{file}]")
        try:
            df = pd.read_csv(file)
            print(f"Rows    : {len(df)}")
            print(f"Columns : {len(df.columns)}")
            print("\nTop 5 Rows")
            display(df.head(5))
        except Exception as ex:
            print(f"Failed to read {file}")
            print(ex)

    print("\n" + "=" * 100)

In [22]:
# run and preview
export_csv(conn)
preview_exports()


[EXPORT] Generating CSV files...
[EXPORT] DynamicPriceOption.csv
[EXPORT] DynamicPriceRange.csv
[EXPORT] CuratedOfferOptions.csv
[EXPORT] Completed

CSV EXPORT PREVIEW

[DynamicPriceOption.csv]
Rows    : 48
Columns : 5

Top 5 Rows


,Provider,OfferId,UniqueOptionId,BestPrice,EnqueuedTimeSP
0,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,b0e296a9-0590-f0e0-8211-243a2ededb12,92.45,18/08/2021
1,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,d6562c24-0b37-5fb4-8275-65b7b8b47b87,92.45,18/08/2021
2,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,8d0f9262-f543-d0c8-a869-33985ae3ecda,92.45,18/08/2021
3,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,151e59ac-761a-96f5-d2b9-882037a9fd28,94.60,18/08/2021
4,ApplyDynamicPricePerOption,56e0702c-0218-4626-8d3d-ae9d54b4503b,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,94.60,18/08/2021



[DynamicPriceRange.csv]
Rows    : 25
Columns : 7

Top 5 Rows


,Provider,OfferId,MinGlobal,MinRecommended,MaxRecommended,DifferenceMinRecommendMinTheory,EnqueuedTimeSP
0,ApplyDynamicPriceRange,a6611d55-9624-4381-8cdd-323ee3689241,85.00,87.20,97.65,2.2,05/09/2021
1,ApplyDynamicPriceRange,b8c636fa-8241-47dc-ac40-bdf438a04d9c,85.00,87.20,97.65,2.2,05/09/2021
2,ApplyDynamicPriceRange,3d32f7fb-396d-4d3f-b673-dea1f7dc41b7,85.00,87.20,97.65,2.2,05/09/2021
3,ApplyDynamicPriceRange,329194f3-95a4-45ef-b3d0-2796f74ce2a0,85.00,87.20,97.65,2.2,05/09/2021
4,ApplyDynamicPriceRange,fdcfde5c-113d-4a59-9ae0-8bc31e2943d8,87.35,89.25,99.95,1.9,05/09/2021



[CuratedOfferOptions.csv]
Rows    : 40
Columns : 18

Top 5 Rows


,CurationProvider,OfferId,DealerId,UniqueOptionId,OptionId,IsMobileDealer,IsOpen,Eta,ChamaScore,ProductBrand,IsWinner,MinimumPrice,MaximumPrice,DynamicPrice,FinalPrice,DefeatPrimaryReason,DefeatReasons,EnqueuedTimeSP
0,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,b0e296a9-0590-f0e0-8211-243a2ededb12,6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2,1,1,1:00,8.0,ULTRAGAZ,1,90.00,180.00,91.90,91.90,NaN,[],25/08/2021
1,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,6517,d6562c24-0b37-5fb4-8275-65b7b8b47b87,6517 || 6517,0,0,0:01,8.0,ULTRAGAZ,0,90.00,180.00,91.90,91.90,Closed,"['Closed', 'HasDriverInOffer']",25/08/2021
2,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,8d0f9262-f543-d0c8-a869-33985ae3ecda,9047 || 9047 || ULTRAGAZ,0,0,1:00,9.0,ULTRAGAZ,0,99.00,198.00,99.95,99.95,Closed,"['Closed', 'HigherPrice', 'HasDriverInOffer']",25/08/2021
3,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,3cd346f4-d297-7568-2e50-d43a8e2fd0a9,9047 || 9047 || CONSIGAZ,0,0,1:00,9.0,CONSIGAZ,0,89.99,179.98,91.89,91.89,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/2021
4,ByPrice,149f0e53-ff85-425f-a01a-8710f06704ea,9047,577e4bbd-f49d-ac23-56a6-e70072a05229,9047 || 9047 || LIQUIGAS,0,0,1:00,9.0,LIQUIGAS,0,92.00,184.00,93.90,93.90,Closed,"['Closed', 'HigherPrice', 'HigherETA']",25/08/2021


============================================DONE===========================================
# Personal Assessment

## Additional Resources Required for Production Deployment
The current solution is implemented as a notebook to validate the Bronze → Silver → Gold transformations. For a production deployment, additional platform resources would be required.

| Capability | Airflow Architecture | Fabric / Azure Architecture | Why Needed in This Solution |
|------------|---------------------|----------------------------|----------------------------|
| Raw Landing Zone | S3 / ADLS / GCS | OneLake / ADLS Gen2 | Store incoming `case.json` events before Bronze ingestion and support replay/recovery. |
| Workflow Orchestration | Apache Airflow DAG | Fabric Data Pipeline / Azure Data Factory | Execute Bronze → Silver → Gold → Export in the correct sequence. |
| Scheduling | Airflow Scheduler | Fabric Pipeline Trigger | Run the pipeline hourly or daily without manual notebook execution. |
| Retry Management | Airflow Task Retry | Pipeline Activity Retry | Automatically retry failed Bronze, Silver, Gold, or Export activities. |
| Workflow Monitoring | Airflow UI | Fabric Monitoring Hub | Track execution status, duration, success/failure rates, and dependency chains. |
| Alerting | Airflow Alerts / Email | Azure Monitor / Fabric Alerts | Notify operators when Silver DQ failures spike or exports fail. |
| Metadata Store | Airflow Variables / Config DB | Config Tables / Lakehouse Metadata | Store schema definitions, DQ rules, and source mappings independently from code. |
| Secrets Management | Vault / AWS Secrets Manager | Azure Key Vault | Store storage credentials and future source-system credentials. |
| Curated Storage | Delta Tables / Warehouse | Lakehouse Tables | Store `bronze_raw_events`, Silver tables, and Gold tables. |
| Export Delivery | Python Task | Notebook Activity | Generate `DynamicPriceOption.csv`, `DynamicPriceRange.csv`, and `CuratedOfferOptions.csv`. |
 


## Scalability Scenarios and Trade-offs
 
| Scenario | Current Approach | Future Scaling Option | Benefits | Trade-offs |
|-----------|-----------------|----------------------|----------|------------|
| Larger JSON files | Single notebook processes entire file | Split input into smaller files and process incrementally | Lower memory usage and faster recovery | More file management complexity |
| Millions of events per day | SQLite + single-node processing | Fabric Lakehouse + Spark + Delta Tables | Distributed processing and horizontal scalability | Higher infrastructure cost |
| Pipeline duration exceeds schedule interval | Single hourly execution | Parallel Bronze, Silver, and Gold activities | Higher throughput and lower latency | Increased orchestration complexity |
| Large Silver tables | Full-table scans | Partition Bronze and Silver by date | Faster querying and aggregations | Additional partition management |
| Gold aggregations become slow | DELETE + INSERT rebuild | Incremental processing using MERGE | Reduced compute time and cost | More complex aggregation logic |
| Very large CSV outputs | Generate entire file in one step | Partitioned exports or Parquet/Delta outputs | Better performance and scalability | Downstream consumers may need format changes |
| Increased number of providers or payload types | Metadata stored in notebook code | External metadata/configuration tables | Easier onboarding of new sources | Additional governance requirements |
| Failures during large loads | Restart entire notebook | Pipeline checkpointing and activity-level reruns | Faster recovery | More operational components to manage |